We'll set everything up here before writing to a .py

In [1]:
# importing our base model
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
import torchvision


/Users/achrafbayi/Desktop/JARVIS-VLA-MINECRAFT/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# I suspect this cell to somehow crash everytime
base_model = Qwen3VLForConditionalGeneration.from_pretrained("Qwen/Qwen3-VL-2B-Instruct", dtype=torch.float32, device_map="cpu")
base_processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct") 
tokenizer = base_processor.tokenizer

Loading weights: 100%|██████████| 625/625 [00:08<00:00, 75.93it/s] 


In [3]:
base_model = base_model.to("mps")

In [5]:
# Ok let's now start working on Phase I which uses the : https://huggingface.co/datasets/CraftJarvis/minecraft-vlp dataset
from huggingface_hub import snapshot_download, hf_hub_download
import huggingface_hub

# here for phase I we just need a single file from the repo
data_step_one = hf_hub_download(repo_id="CraftJarvis/minecraft-vlp", filename="mc-qa-250312.jsonl", repo_type="dataset", local_dir="./data_phase_one")


# we want to train to predict : P(assistant | system, user)

In [6]:
# for reference here is what one line of that json looks like :

'''
{"id": "qa-f89343ef-d687-49b5-95d5-1d0e9d63da18", "label": ["qa", "wiki", "self-instruct"], "image": [], 
"conversations": 
[{"role": "system", "content": [{"type": "text", "text": "You are a helpful assistant that can answer questions and help with tasks in Minecraft.\n"}]}, 
{"role": "user", "content": [{"type": "text", "text": "How do you enchant items in Minecraft, and what are some of the enchantments that can be applied?"}]}, 
{"role": "assistant", "content": [{"type": "text", "text": "You can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune."}]}]}


No image context for phase 1 ofc
'''

#Let's real quickly try to parse that line fine before doing it on the whole big ass json file
import json 

with open(data_step_one) as f:
    lines = f.readlines()

line = lines[12]
#print(line)

dict = json.loads(line)
dict["conversations"][1]["content"][0]['text'] # How do you enchant items in Minecraft, and what are some of the enchantments that can be applied? 
# the question
system=dict["conversations"][0]["content"][0]['text']
user = dict["conversations"][1]["content"][0]['text']
assistant = dict["conversations"][2]["content"][0]['text']

# format we want (with the Qwen tokens/markers and the format it was pretrained on)

pre_tokenization_datapoint = '<|im_start|>system\n'+system+'<|im_end|>\n'+'<|im_start|>user\n'+user+'<|im_end|>\n'+'<|im_start|>assistant\n'+assistant+'<|im_end|>\n'
print(pre_tokenization_datapoint)

'''
Good :

<|im_start|>system
You are a helpful assistant that can answer questions and help with tasks in Minecraft.
<|im_end|>
<|im_start|>user
How do you enchant items in Minecraft, and what are some of the enchantments that can be applied?<|im_end|>
<|im_start|>assistant
You can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune.<|im_end|>
'''



<|im_start|>system
You are a helpful assistant that can answer questions and help with tasks in Minecraft.
<|im_end|>
<|im_start|>user
How do you enchant items in Minecraft, and what are some of the enchantments that can be applied?<|im_end|>
<|im_start|>assistant
You can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune.<|im_end|>



'\nGood :\n\n<|im_start|>system\nYou are a helpful assistant that can answer questions and help with tasks in Minecraft.\n<|im_end|>\n<|im_start|>user\nHow do you enchant items in Minecraft, and what are some of the enchantments that can be applied?<|im_end|>\n<|im_start|>assistant\nYou can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune.<|im_end|>\n'

In [7]:
# now that we got the format right, let's do it over the whole json, and store it

with open('./data_phase_one/phase_one_formatted.jsonl', 'x') as f: # exclusive for no re-write unless manual deletion of file
    for line in lines:
        dict = json.loads(line)
        system=dict["conversations"][0]["content"][0]['text']
        user = dict["conversations"][1]["content"][0]['text']
        assistant = dict["conversations"][2]["content"][0]['text']
        text = '<|im_start|>system\n'+system+'<|im_end|>\n'+'<|im_start|>user\n'+user+'<|im_end|>\n'+'<|im_start|>assistant\n'+assistant+'<|im_end|>\n'
        f.write(json.dumps({'datapoint':text})+'\n')


FileExistsError: [Errno 17] File exists: './data_phase_one/phase_one_formatted.jsonl'

In [8]:
# now let's set up our proper dataset
# we want to keep the sentences separate, we hence want sumn like an array of arrays of tokens
import json 

with open('./data_phase_one/phase_one_formatted.jsonl', 'r') as f:
    buckets = [json.loads(line)['datapoint'] for line in f.readlines()]

#print(buckets[2]) we get what we want

token_buckets = [tokenizer(point, add_special_tokens=False) for point in buckets]
print(len(token_buckets))

print(token_buckets[3]) # good, we get the token ids

275905
{'input_ids': [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 429, 646, 4226, 4755, 323, 1492, 448, 9079, 304, 40230, 624, 151645, 198, 151644, 872, 198, 4340, 653, 498, 1855, 264, 3731, 10812, 16224, 304, 40230, 323, 1128, 525, 1045, 315, 279, 2155, 5711, 369, 432, 30, 151645, 198, 151644, 77091, 198, 641, 40230, 11, 3731, 10812, 374, 264, 4911, 5101, 429, 646, 387, 1483, 311, 1855, 6351, 45021, 323, 7611, 13, 3731, 10812, 374, 1483, 311, 29282, 2355, 11, 20276, 23783, 11, 323, 68611, 5257, 9079, 304, 279, 1809, 13, 5692, 594, 264, 6770, 8474, 389, 1246, 311, 1855, 264, 3731, 10812, 16224, 323, 1045, 2155, 5711, 369, 432, 1447, 334, 16, 13, 48995, 3731, 10812, 25, 1019, 256, 481, 2014, 1191, 11, 498, 1184, 311, 10485, 3731, 10812, 16144, 10010, 11, 892, 525, 5990, 1730, 5538, 25260, 13, 1446, 3278, 1184, 458, 11001, 3735, 49739, 476, 2664, 311, 6530, 3731, 10812, 15797, 382, 334, 17, 13, 4230, 3731, 10812, 31993, 25, 1019, 256, 481, 9646, 498, 614, 3731, 10812, 16144, 11, 104

In [ ]:
# ok, now that we have that, we want to build our dataset for GPT-like training, where
# for each x, y=x+1 (in token indexing)
# but here we also don't want to train the llm to predict the user and system tokens
# hence we need our shortest substring to be the following : <System content and tokens><User content and tokens><|im_start|>assistant\n
# and build on from that to the last token
# the string that delimits (and that is unique in each datapoint which we can use as the "border") in every datapoint is the following
# <|im_start|>assistant\n
# let's tokenize it to make it the magic tokens that signal the start of or sequence
start_tokens = tokenizer("<|im_start|>assistant\n", add_special_tokens=False)['input_ids']
print(start_tokens)
# [151644, 77091, 198] those 3 are the start tokens, hence we'd want to look for those 3 tokens before the start of each of our sequences

magic_tokens = [151644, 77091, 198]
dsx_idx = []
for datapoint, n in zip(token_buckets, range(len(token_buckets))):
        # we'll use sliding window on each sequence to detect starting index
        datapoint = datapoint['input_ids']
        x = -1
        length = len(datapoint)
        for i in range(length-2):
            if datapoint[i] == 151644 and datapoint[i+1] == 77091: # our magic tokens
                if datapoint[i+2]==198: # i.e the token for \n, meaning it didn't get merged into a bigger token, we skip it
                    x = i+3 # the convention we'll use here is that x stores the ds[:x] for x in range(x, len-1)
                    # and here the convention will be that y = ds[x] simply as the next token we want to preditct
                else:
                    x = i+2
        if x==-1 : 
            print(x)
            print(n) # to know which line is dirty 
            print(buckets[n])
            print(token_buckets[n])
            raise RuntimeError("we have empty x's and y's")
        else: dsx_idx.append(x)
'''
Btw here, the reason why we did not include the 'and datapoint[i+2] == 198' in our conditional is that we encountered cases where that \ n char 
got merged into another 'bigger/newer' token (in the BPE sense) and hence was missing and threw the error, hence that (squence at index 17067)

if x==[] or y==[]: 

is a really good guard as it allows us to stall whenever something like that happens (initially we had the and datapoint[i+2] == 198 component in our
sliding window boolean)

'''

# checking if 17067 is now fixed :

print(tokenizer.decode(token_buckets[17067]['input_ids'][dsx_idx[17067]+1:], add_special_tokens=False)) # do indeed get the right answer which is reassuring


<>:35: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:35: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
/var/folders/13/1fmmyq5j5n303885dkdjlzn00000gn/T/ipykernel_13184/3735964709.py:35: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  Btw here, the reason why we did not include the 'and datapoint[i+2] == 198' in our conditional is that we encountered cases where that \ n char


[151644, 77091, 198]
Minecraft offers endless possibilities for creative building ideas that utilize redstone. Redstone is a versatile resource that can be used to create complex contraptions, automation systems, and interactive structures. Here are some creative building ideas to get you started:

1. **Hidden Entrance or Base:** Create a hidden entrance to your base using redstone mechanisms like piston doors, secret levers, or pressure plates. Combine them with redstone lamps or note blocks for added effects.

2. **Automated Farm:** Build an automated farm that uses redstone to harvest and replant crops, such as wheat, carrots, or potatoes. You can also create a fully automatic tree farm or animal breeding system.

3. **Elevator or Escalator:** Design a redstone-powered elevator or escalator that allows you to move between different floors of your building effortlessly. Elevators can be especially impressive when designed with multiple floors and destinations.

4. **Adventure Map or 

In [29]:
# we basically now have with token buckets and dsx_idx the right dataset and the indx on which to start training
import numpy as np
ds = list(zip(dsx_idx, (tok['input_ids'] for tok in token_buckets)))

import random
random.Random(67).shuffle(ds) #shuffling for sgd in case the user query -> responses were in some kind of order, and to split cleanly

breaker = int(len(ds)*0.99) # we only keep 1 percent for test set this time

train_ds = ds[:breaker]
test_ds = ds [breaker:]
print(len(test_ds))
print(len(max(ds, key=lambda x :len(x[1]))[1])) # below the paper's 3584 for phase I
# below the paper's 3584 for phase I
print(np.mean([len(x[1]) for x in ds])) # mean is 402
# so no need to ever cut a sequence

2760
1672
411.4454323045976


In [ ]:
print(tokenizer.pad_token_id)
print(tokenizer.decode(tokenizer.pad_token_id)) # it is the EOF token ?

151643
<|endoftext|>


In [ ]:
# Phase I Training loop :

# first of all, we need to freeze all the ViT/vision related parameters of our Qwen VLM
import math

for p in base_model.model.visual.parameters():
    p.requires_grad = False


# then we set up the optimizer, learning rate and everything, we only want to pass in the parameters that have required_grad = True ofc
# as well as the parameters specified by the paper (I think default for AdamW B2 is like 0.999 so we do need to pass em explicitly)
trainable_params = [p for p in base_model.parameters() if p.requires_grad == True]
optimizer = torch.optim.AdamW(
    params = trainable_params,
    betas=(0.9, 0.95),
    eps=1e-8,
    lr=5e-6,
    weight_decay=0
)

# let's define our max_steps, since the paper doesn't specifiy the number of steps/epochs they used, I'll assume 1 epoch
print(len(train_ds))
B = 256
max_step = int(len(train_ds)/B) #273145

# now our cosine lr w/ warmup function we'll use in the loop
lr_max = 5e-6
warmup_steps = 200
# warmup of 200 steps
def get_lr(step):
    # here the paper didn't specify lr_min so for the function I'll just assume it to be 0
    if step <= warmup_steps:
        return step * (lr_max/200)
    else:
        return (1/2)*lr_max*(1+math.cos(((step-warmup_steps)/max_step)*math.pi))

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu'
import torch.nn.functional as F

# actually rather than building the batch inside the iters, we might want to make it into a function, maybe for later use
eval_every = 150

def build_batch_tensors(ds):
    max_seq_length = len(max(ds, key=lambda x : len(x[1]))[1]) -2 # -1 since last 2 tokens is never appart of any inputs
    # we want the longest sequence to be our shared seq length dimension across all sequence 3d tensors, and we'll add padding to all ones that don't meet that max
    # we want to get the pad token as I am not sure which one it is for qwen, we'll print it above, ok it is actually the EOF token so let's now build our batched tensor
    # and feed that bitch to QwenVL's llm
    x_batch_tensor = torch.empty(size=(len(ds), max_seq_length), dtype=torch.int64)
    y_batch_tensor = torch.empty(size=(len(ds), max_seq_length), dtype=torch.int64) # using int64 as our token ids are int64
    for tup, k in zip(ds, range(len(ds))):
        x_cat = torch.tensor(data=tup[1][:-2]) # (1,len) # skiping the end and \n last 2 characters
        pad = torch.full(size=(max_seq_length-len(x_cat),), fill_value=tokenizer.pad_token_id)
        x = torch.cat([x_cat, pad], dim=0)
        y_cat = torch.full(size=(tup[0]-1,), fill_value=-67) # out of range, negative
        actual_values = torch.tensor(data=[tup[1][w] for w in range(tup[0], len(tup[1])-1)])  # in range, the actual ground truth
        y_pad = torch.full(size=(max_seq_length - (len(tup[1])-2),), fill_value=-67) # padding with our future ignore index
        y = torch.cat([y_cat,actual_values,y_pad], dim=0)
        x_batch_tensor[k] = x
        y_batch_tensor[k] = y
    return x_batch_tensor,y_batch_tensor

# we'll also want to log our loss to draw nice curves
import os, time
run_id = time.strftime("%Y%m%d-%H%M%S")
log_dir = f"phase_one_logs/{run_id}"
os.makedirs(log_dir, exist_ok=True)

train_log = open(f"{log_dir}/train.csv", "w"); train_log.write("step,loss,lr\n")
test_log  = open(f"{log_dir}/test.csv", "w");  test_log.write("step,loss\n")

# we'll use this function to compute test loss
@torch.no_grad()
def test_loss(bs=16):
    base_model.eval()
    total, n_tok = 0.0, 0
    for i in range(0, len(test_ds), bs):
        x, y = build_batch_tensors(test_ds[i:i+bs])
        x, y = x.to(device), y.to(device)
        logits = base_model(input_ids=x).logits
        total += F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=-67, reduction="sum").item()
        n_tok += (y != -67).sum().item()
    base_model.train()
    return total / n_tok


# as said in the paper we want 256-long batches (256 sequences)
# hence that will require 256 forward passes of the llm, and multiplex into 256*lenght of the sequences (post-system+user content) loss computations
base_model.train() # flipping the value
for step in range(max_step):
    # we need to set up the batch
    # let's remember that our ds is in the form [(idx, tokens)] where ids is where the assistant part of the sequence beings
    batch_ds = train_ds[(step)*B:(step+1)*B] # start is inclusive, end exclusive
    x, y = build_batch_tensors(batch_ds)

    # now that we have our batched tensor, let's push them through the llm of Qwen3VL

    x = x.to(device)
    y = y.to(device)
    optimizer.zero_grad() # zeroing out the gradient values first
    out = base_model(input_ids=x) # going from token_ids to the actual vectors happens in the model so we're gucci
    logits = out.logits # we get something of the shape (B, len, vocab) i.e one logit distribution per token which is what we wanted
    loss = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]), # (B*len, vocab)
        y.reshape(-1), # (B*len)
        ignore_index=-67
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0) # gradient clipping
    for g in optimizer.param_groups: g["lr"] = get_lr(step)
    optimizer.step()

    train_log.write(f"{step},{loss.item()},{get_lr(step)}\n"); train_log.flush()
    if step % 50 == 0:
        print(f"step {step} | loss {loss.item():.4f} | lr {get_lr(step):.2e}")

    if step % eval_every == 0 or step == max_step - 1: # test loss every 200
        tl = test_loss()
        test_log.write(f"{step},{tl}\n"); test_log.flush()
        print(f"  test loss {tl:.4f}")

    if step == max_step-1: # since phase one is kind of short, it is ok to only save at the end
        base_model.save_pretrained(f"checkpoints/{run_id}/phase1")




273145


In [9]:
# now starting with phase 2
from huggingface_hub import snapshot_download

# first off we need to unfreeze all the weights

for p in base_model.parameters():
    p.requires_grad = True
    

    # now we need to set up the visual tokens + instruction part, and we need to import it from the dataset

    data_step_two = snapshot_download(
        repo_id="CraftJarvis/minecraft-vlp",
        repo_type="dataset",
        local_dir="./data_phase_two", 
        ignore_patterns=["mc-knowledge-valid.jsonl", "mc-qa-250312.jsonl", "hallucination.jsonl", ".gitattributes"]) # ignoring the phase 1 stuff


Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 784.88it/s]


In [ ]:
# now that I have downloaded the data, we get to see the structure of it, we have 5 jsonl files with {"id": "6940125a-15fc-4776-9dc5-ae75cf0f4949", "label": ["vqa", "contractor", "Minecraft"], "image": ["images/mc-vqa-241102/6940125a-15fc-4776-9dc5-ae75cf0f4949.jpg"], "datetime": "24_11_02", "model": "gpt_4o", "conversations": [{"role": "user", "content": [{"type": "text", "text": "What type of bed is present in the image?\n"}, {"type": "image"}]}, {"role": "assistant", "content": [{"type": "text", "text": "The bed in the image is a yellow bed. In Minecraft, beds are used for sleeping to skip to dawn and to set a player's respawn point. Beds come in various colors, and the color is determined by the dye used in crafting. The yellow bed is crafted using yellow dye and wool.\n"}]}]}

#from mc-vqa-241102.jsonl
'''
{"id": "6940125a-15fc-4776-9dc5-ae75cf0f4949", 
"label": ["vqa", "contractor", "Minecraft"], 
"image": ["images/mc-vqa-241102/6940125a-15fc-4776-9dc5-ae75cf0f4949.jpg"], "datetime": "24_11_02", 
"model": "gpt_4o", "conversations": 
[{"role": "user", "content": [{"type": "text", "text": "What type of bed is present in the image?\n"}, {"type": "image"}]}, 
{"role": "assistant", "content": [{"type": "text", "text": "The bed in the image is a yellow bed. In Minecraft, beds are used for sleeping to skip to dawn and to set a player's respawn point. Beds come in various colors, and the color is determined by the dye used in crafting. The yellow bed is crafted using yellow dye and wool.\n"}]}]}
'''

# as we can see, we get a file path (relative) the user prompt (no system prompt) and the assistant output we're trying to predict
# btw this sample is from mc-vqa ... .jsonl , let's see if other formats differ in any other ways
# they actually vary in format which sucks as it means more code

# from mc-grounding-point-gui.jsonl
'''
{"id": "0de6ae27-3dce-4e21-83fa-10d7f85212e1", 
"label": ["gui", "Minecraft", "point"], "image": ["images/mc-grounding-point-gui/0de6ae27-3dce-4e21-83fa-10d7f85212e1.jpg"], 
"conversations": 
[{"role": "user", "content": [{"type": "text", "text": "Point directly to the exact location of the recipe book in the image.\n"}, 
{"type": "image", "text": "<image>"}]}, 
{"role": "assistant", "content": [{"type": "point", "point": [[79.1, 40.1]], "label": "recipe book"}]}], 
"source": {"bbox": [[415, 72], [431, 86]], "image_source": "/public/hkc/craft_cold_start/craft_inv/frames_valid/640/frame_119.jpg"}}
'''
# here the format seems to be user prompt + image tokens, and then we try to predict the points

# mc-grounding-point-embodied.jsonl
'''
{"id": "No.Grounding.6xx.Player261-f153ac423f61-20211022-162743.393.chest", 
"label": ["embody", "Minecraft", "point"], 
"conversations": 
[{"role": "user", "content": [{"type": "text", "text": "Pinpoint the chest.\n"}, 
{"type": "image", "text": "<image>"}]}, 
{"role": "assistant", "content": [{"type": "point", "point": [[82.1875, 83.88888888888889]], "label": "chest"}]}], "datetime": "2025-02-21 12:50:19", "model": "SAM2", 
"image": ["images/mc-grounding-point-embodied/6xx-Player261-f153ac423f61-20211022-162743-393.jpg"], 
"source": {"image_url": "images/grounding-event-sam2-0221/6xx-Player261-f153ac423f61-20211022-162743-393.jpg", 
"video_path": "/nfs-shared/data/contractors/all_6xx_Jun_29/videos/Player261-f153ac423f61-20211022-162743.mp4",
 "points": [302, 526], "bbox": [409, 236, 639, 359], "label": "chest"}}
'''
# this one is also of the format user prompt + image -> single point prediction, so similar to the one above 

#mc-grounding-point-embodied-image5.jsonl

'''
{"id": "No.Grounding.6xx.Player670-f153ac423f61-20210810-105232.798.800.evoker", 
"label": ["embody", "Minecraft", "point"], "conversations": [
{"role": "user", "content": [{"type": "text", "text": "Pin down the evoker in the image, providing its location in points.\n"}, 
{"type": "image", "text": "<image>"}]}, 
{"role": "assistant", "content": [{"type": "point", "point": [[53.75, 42.77777777777778]], "label": "evoker"}]}, 
{"role": "user", "content": [{"type": "image", "text": "<image>"}]}, 
{"role": "assistant", "content": [{"type": "point", "point": [[51.87500000000001, 47.22222222222222]], "label": "evoker"}]}], 
"datetime": "2025-02-21 17:36:36", "model": "SAM2", "image": ["images/mc-grounding-point-embodied-image5/6xx-Player670-f153ac423f61-20210810-105232-798.jpg", "images/mc-grounding-point-embodied-image5/6xx-Player670-f153ac423f61-20210810-105232-800.jpg"], 
"source": {"image_urls": ["images/grounding-event-sam2-image=5-0222/6xx-Player670-f153ac423f61-20210810-105232-798.jpg", "images/grounding-event-sam2-image=5-0222/6xx-Player670-f153ac423f61-20210810-105232-800.jpg"], 
"video_path": "/nfs-shared/data/contractors/all_6xx_Jun_29/videos/Player670-f153ac423f61-20210810-105232.mp4", 
"points": [[154, 344], [170, 332]], "bbox": [[220, 63, 437, 239], [216, 80, 416, 255]], "label": "evoker", "action": "kill_entity:evoker"}}
'''

# again here, we also seem to have some kind of user prompt + image tokens -> point

#mc-caption-241104.jsonl

'''
{"id": "88140ef9-3c2e-4abb-abf0-585799c725e3", 
"label": ["caption", "contractor", "question_based", "Minecraft"], 
"image": ["images/mc-caption-241104/88140ef9-3c2e-4abb-abf0-585799c725e3.jpg"], "datetime": "24_11_04", "model": "gpt_4o", 
"conversations": [
{"role": "user", "content": [{"type": "text", "text": "Discuss the context and content visible in this Minecraft scene.\n"}, 
{"type": "image"}]}, 
{"role": "assistant", "content": [{"type": "text", "text": "This Minecraft image is taken from a first-person perspective, offering an immersive view of the game's blocky world. The scene unfolds on a hilly terrain, where natural elements like grass and dirt meet the player's modifications. A path made of wooden planks leads up the hill, suggesting previous player activity in shaping the landscape. The area where the dirt, grass, and cobblestone meet is the main focal point, possibly indicating a path or structure that has been built or altered by the player. The mix of textures, including cobblestone, dirt, grass, and wooden planks, adds depth to the scene. The landscape is a blend of natural generation and player alteration, with the wooden plank paths and cobblestone structures standing out against the naturally occurring grass and dirt.\n"}]}]}
'''

# here its descriptive work, still in user, image assistant

#o verall all 5 seem to all have the user + image tokens -> assistant ground truth, and though the semantics/nature of the stuff
# we want to predict varies, it is always the same format meaning we can build a "global" function to handle that data 
# (we'll do so without caching the processed data this time)

In [28]:
# testing rq
import json

with open("./data_phase_two/mc-caption-241104.jsonl", "r") as f:
    data = json.loads(next(f)) # we just want one line

user = data['conversations'][0]['content'][0]['text']
assistant = data['conversations'][1]['content'][0]['text']
image_path = data['image'][0]

print(f'{user}\n{assistant}\n{image_path}') 

# this is the right query format for the conversation one
# promlem is that the grounding one has a different format in the conversation array


Please provide a caption for this Minecraft screenshot.

In the heart of a dimly lit cave, a tense and adventurous scene unfolds. The main focal point is a zombie, positioned centrally and directly in front of the player, its presence contributing to the overall sense of danger. The surrounding environment is sparse, with stone walls that focus attention on the immediate threat. As the zombie approaches, the area immediately around it becomes the focal point, drawing the player's attention to the impending hostile encounter.

The cave's landscape, typical of an underground system, features a visible lava pool that not only poses a hazard but also serves as a recognizable landmark for navigation. The presence of lava and stone walls suggests that this area may offer resource-gathering opportunities, such as mining for stone or possibly valuable ores like diamonds or obsidian. However, the current atmosphere is one of tension and survival, rather than exploration or resource collection.


In [ ]:
# we want a function that takes our jsons and returns both the x (to which we'll concat the image tokens) and our y
import json 

def visual_ds_build(jsonl_path):
    with open(jsonl_path, "r") as f:
        data = [json.loads(line) for line in f] # every line is a json doc
    x, y = [],[] # here is the format we'll want :
    # x : tuple, (formatted token array of the prompt, corresponding image file path)
    # y : array of array of tokens as for ground truth (formatted)
    for instance in data:
        user = instance['conversations'][0]['content'][0]['text']
